# Day 04 — Merges and Joins
**Estimated time:** 60–75 min  
**Datasets:** `sales_orders.csv`, `materials_inventory.csv`

## Learning Objectives
- Execute inner, left, right, and outer joins using `pd.merge()`
- Diagnose join quality: row count changes, duplicate keys, null introduction
- Identify many-to-many join risks (the silent data explosion)


In [ ]:
import pandas as pd
import numpy as np

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
sales["ERDAT"] = pd.to_datetime(sales["ERDAT"], errors="coerce")
sales["NETWR"] = pd.to_numeric(sales["NETWR"], errors="coerce")

inv = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
inv["LABST"] = pd.to_numeric(inv["LABST"], errors="coerce")
inv["STPRS"] = pd.to_numeric(inv["STPRS"], errors="coerce")

# MATNR cardinality check — critical before joining
print(f"Sales MATNR unique: {sales['MATNR'].nunique():,} | total rows: {len(sales):,}")
print(f"Inv   MATNR unique: {inv['MATNR'].nunique():,} | total rows: {len(inv):,}")


In [ ]:
# ── 1. Inner join — only matched records ─────────────────────────────────────
inner = pd.merge(sales, inv, on="MATNR", how="inner", suffixes=("_sales","_inv"))
print(f"Inner join rows:     {len(inner):,}")
print(f"Sales rows:          {len(sales):,}")
print(f"Inventory rows:      {len(inv):,}")
print(f"Row ratio (inner/sales): {len(inner)/len(sales):.2f}x  ← >1.0 means many-to-many!")


In [ ]:
# ── 2. Left join — keep all sales, attach inventory info ─────────────────────
left = pd.merge(sales, inv, on="MATNR", how="left", suffixes=("_sales","_inv"))
print(f"Left join rows: {len(left):,}")

# Null analysis: which sales orders have no inventory record?
no_inv_match = left[left["LABST"].isna()]
print(f"Sales orders with no inventory match: {len(no_inv_match):,} ({len(no_inv_match)/len(left):.1%})")
print(no_inv_match[["VBELN","MATNR","NETWR"]].head())


In [ ]:
# ── 3. Diagnose many-to-many risk ────────────────────────────────────────────
# If inv has multiple rows per MATNR (e.g., per plant), joining will explode rows
inv_matnr_dupes = inv.groupby("MATNR").size()
multi_plant_materials = inv_matnr_dupes[inv_matnr_dupes > 1]
print(f"MATNRs with multiple inventory records: {len(multi_plant_materials):,}")
print(f"Max records per MATNR: {inv_matnr_dupes.max()}")

# Solution: aggregate inventory to single row per MATNR before joining
inv_agg = (inv.groupby("MATNR")
              .agg(
                  total_labst=("LABST","sum"),
                  total_inv_value=pd.NamedAgg("STPRS", lambda x: (x * inv.loc[x.index,"LABST"]).sum()),
                  plant_count=("WERKS","nunique"),
                  material_type=("MATERIAL_TYPE","first"),
                  maktx=("MAKTX","first"),
              )
              .reset_index())
print(f"\nAggregated inventory rows: {len(inv_agg):,}  (one per MATNR)")


In [ ]:
# ── 4. Clean join: sales + aggregated inventory ───────────────────────────────
%%time
merged = pd.merge(sales, inv_agg, on="MATNR", how="left")
print(f"Merged shape: {merged.shape}")
print(f"Row count == sales? {len(merged) == len(sales)}")  # must be True for 1:1
display(merged.head())


In [ ]:
# ── 5. Outer join — diagnose unmatched on both sides ──────────────────────────
outer = pd.merge(sales[["MATNR","VBELN","NETWR"]].drop_duplicates("MATNR"),
                 inv_agg[["MATNR","total_labst"]],
                 on="MATNR", how="outer", indicator=True)

print(outer["_merge"].value_counts())
# both         = matched
# left_only    = materials in sales but not in inventory master
# right_only   = materials in inventory but never sold


In [ ]:
# ── 6. Validate join with a post-merge null scan ─────────────────────────────
def join_quality_report(df_merged, key_col, left_cols, right_cols):
    report = {}
    report["total_rows"] = len(df_merged)
    for col in right_cols:
        null_pct = df_merged[col].isna().mean()
        report[f"{col}_null_pct"] = f"{null_pct:.1%}"
    return pd.Series(report)

report = join_quality_report(merged, "MATNR", ["VBELN","NETWR"], ["total_labst","material_type"])
print(report)


---
## Daily Prompt

**Dead stock analysis: Which materials in the inventory master had NO sales orders in the last 12 months?**

These materials tie up working capital with no demand signal. Report: MATNR, MAKTX, plant, total stock quantity, inventory value, last movement date.

```python
# YOUR SOLUTION
cutoff_12m = pd.Timestamp.today() - pd.DateOffset(months=12)

# Step 1: find MATNRs with recent sales
recent_sold_matns = (
    sales.loc[sales["ERDAT"] >= cutoff_12m, "MATNR"]
    .unique()
)

# Step 2: YOUR SOLUTION — filter inventory for materials NOT in recent_sold_matns
dead_stock = inv.loc[
    ~inv["MATNR"].isin(recent_sold_matns)
    # AND has actual stock > 0
]

# Step 3: calculate inventory value and sort
```

> **Hint:** `~inv["MATNR"].isin(recent_sold_matns) & (inv["LABST"] > 0)`.  
> Add `inv_value = inv["LABST"] * inv["STPRS"]` and sort descending.  
> What is the total capital at risk? `dead_stock["inv_value"].sum()`
